# IndicWrite — genuine BharatGen smoke test
Pretrained inference remains UNVERIFIED until this notebook produces real logs. Select a T4 GPU runtime. Download is about 5.72 GB of weight shards, plus tokenizer and packages. Use a disposable runtime with no Drive mount or secrets. Review the pinned model Python files before enabling remote code. Runtime success is not a meaning-preservation score.

Run this notebook from the coordinator's integrated commit. Set `SOURCE_COMMIT` below to that full SHA (not the pre-patch base commit). Source review base: `9a5ca96b5ea7a66bf3c90fe8ea9dee50faad0f71`. Model revision: `f9b32a727458727ff515fea5d82df00ae9b7ad56`.

**Immutable source revision prepared by coordinator:** `ef1cc3bd9457007f6adb15769630cfed97e82bc4`. Only the official checkpoint and research scripts from this specific commit will run; no GitHub write permission is needed for the Colab experiment.

**Colab compatibility update:** Your notebook kernel may use Python 3.11 or 3.12; the setup now installs an isolated Python 3.12 with `uv`. No runtime Python switching is required. Even on errors, the evidence ZIP can be downloaded from the final cell.


In [ ]:
import pathlib, re, subprocess, sys
SOURCE_COMMIT = "ef1cc3bd9457007f6adb15769630cfed97e82bc4"
assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT), "Set the coordinator's integrated source commit SHA."
print("Notebook kernel Python:", sys.version.split()[0], "| project will use isolated Python 3.12.")
ROOT = pathlib.Path('/content/indicwrite-bharatgen-' + SOURCE_COMMIT[:12])
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Rishikeshsanin/Hinglish-Smart-Writing-Assistant.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=ROOT, check=True)
assert 'MODEL_REVISION' in (ROOT/'src/indicwrite/param_rewrite.py').read_text(), "The selected commit lacks the review patch."
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Colab's kernel Python may be 3.11/3.12. Use a separate Python 3.12 via uv.
import shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv>=0.9,<1'], check=True)
UV = shutil.which('uv')
assert UV, 'uv was not installed into the kernel PATH'
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
VENV = pathlib.Path('/content/indicwrite-bharatgen-venv')
subprocess.run([UV, 'venv', '--python', '3.12', '--seed', str(VENV)], check=True)
PY = str(VENV/'bin/python')
subprocess.run([PY, '-c', 'import sys; print(sys.version); assert sys.version_info[:2] == (3,12)'], check=True)
# Pinned CUDA runtime on disposable GPU; do not change the Colab kernel.
subprocess.run([PY, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([PY, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-bharatgen.txt')], check=True)
subprocess.run([PY, '-m', 'pip', 'check'], check=True)
subprocess.run([PY, '-c', "import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())"], check=True)


Review these pinned files as text before continuing:
- https://huggingface.co/bharatgenai/Param-1-2.9B-Instruct/blob/f9b32a727458727ff515fea5d82df00ae9b7ad56/config_parambharatgen.py
- https://huggingface.co/bharatgenai/Param-1-2.9B-Instruct/blob/f9b32a727458727ff515fea5d82df00ae9b7ad56/modeling_parambharatgen.py

The loader checks their SHA-256 values before importing them. This is identity checking, not a sandbox. The card has inconsistent licensing statements and no standalone LICENSE at this revision; preserve that unresolved finding in the handoff.


In [ ]:
from datetime import datetime, timezone
RUN = ROOT/'evaluation'/'results'/('bharatgen-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
RUN.mkdir(parents=True)
(RUN/'pip-freeze.txt').write_text(subprocess.check_output([PY, '-m', 'pip', 'freeze'], text=True))
(RUN/'nvidia-smi.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True))
with (RUN/'console.log').open('w') as console:
    result = subprocess.run(
        [PY, '-u', 'scripts/smoke_bharatgen.py', '--allow-model-download',
         '--allow-remote-code', '--output', str(RUN/'smoke.jsonl')],
        cwd=ROOT, stdout=console, stderr=subprocess.STDOUT
    )
print((RUN/'console.log').read_text())
SMOKE_OK = result.returncode == 0
print("Actual pretrained smoke succeeded:", SMOKE_OK)
if not SMOKE_OK:
    print("Do NOT claim pretrained inference succeeded. Download evidence ZIP from last cell.")


In [ ]:
# Run only after the one-case smoke succeeds. References are never fed into the model.
if not SMOKE_OK:
    print('Skipping 10-case experiment because the genuine pretrained smoke failed.')
else:
    with (RUN/'ten-cases-console.log').open('w') as console:
        result = subprocess.run(
            [PY, '-u', 'scripts/smoke_bharatgen.py', '--allow-model-download',
             '--allow-remote-code', '--cases', 'evaluation/hindi_rewrite_cases.json',
             '--output', str(RUN/'ten-cases.jsonl')],
            cwd=ROOT, stdout=console, stderr=subprocess.STDOUT
        )
    print((RUN/'ten-cases-console.log').read_text())
    print('Ten-case execution succeeded:', result.returncode == 0)
print('Evidence directory:', RUN)
print('Human meaning-preservation review is still required.')


In [ ]:
# Download actual evidence even if generation failed. Never package weights or caches.
import shutil
from google.colab import files
archive = shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
files.download(archive)
